In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/content/amazon.csv")

df.head()

,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,keys,...,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,sizes,upc,weight
0,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I initially had trouble deciding between the p...,"Paperwhite voyage, no regrets!",NaN,NaN,Cristina M,NaN,NaN,205 grams
1,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,Allow me to preface this with a little history...,One Simply Could Not Ask For More,NaN,NaN,Ricky,NaN,NaN,205 grams
2,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,4.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I am enjoying it so far. Great for reading. Ha...,Great for those that just want an e-reader,NaN,NaN,Tedd Gardiner,NaN,NaN,205 grams
3,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I bought one of the first Paperwhites and have...,Love / Hate relationship,NaN,NaN,Dougal,NaN,NaN,205 grams
4,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I have to say upfront - I don't like coroporat...,I LOVE IT,NaN,NaN,Miljan David Tanic,NaN,NaN,205 grams


In [18]:
print(df['sentiment'].value_counts(normalize=True))

sentiment
1.0    0.927825
0.0    0.072175
Name: proportion, dtype: float64


In [4]:
# Keep only required columns
df = df[['reviews.text','reviews.rating']]

# Drop missing values
df = df.dropna()

# Convert rating → sentiment
def convert_label(x):
    if x >= 4:
        return 1
    elif x <= 2:
        return 0
    else:
        return None

df['sentiment'] = df['reviews.rating'].apply(convert_label)

# Remove neutral reviews
df = df.dropna()

texts = df['reviews.text'].astype(str).tolist()
labels = df['sentiment'].astype(int).values

print("Total samples:", len(texts))

Total samples: 1053


In [19]:
df_pos = df[df['sentiment'] == 1]
df_neg = df[df['sentiment'] == 0]

min_len = min(len(df_pos), len(df_neg))

df = pd.concat([
    df_pos.sample(min_len, random_state=42),
    df_neg.sample(min_len, random_state=42)
]).sample(frac=1, random_state=42)

print(df['sentiment'].value_counts())

sentiment
1.0    76
0.0    76
Name: count, dtype: int64


In [20]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE = 10000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=VOCAB_SIZE)
tokenizer.fit_on_texts(texts)

seqs = tokenizer.texts_to_sequences(texts)
X = pad_sequences(seqs, maxlen=MAX_LEN)

X = torch.tensor(X)
y = torch.tensor(labels)

In [21]:
split = int(0.8*len(X))

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [22]:
class GRUModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE,128)
        self.gru = nn.GRU(128,128,batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(128,1)

    def forward(self,x):
        x = self.emb(x)
        out,_ = self.gru(x)
        out = self.dropout(out[:,-1])
        return torch.sigmoid(self.fc(out))

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GRUModel().to(device)
loss_fn = nn.BCELoss()
opt = optim.Adam(model.parameters(),0.001)

# LIMIT DATA FOR SPEED
LIMIT = min(10000, len(X_train))

for epoch in range(5):
    correct = 0
    total = 0

    for i in range(0, LIMIT, 64):
        xb = X_train[i:i+64].to(device)
        yb = y_train[i:i+64].float().unsqueeze(1).to(device)

        opt.zero_grad()
        out = model(xb)
        loss = loss_fn(out,yb)
        loss.backward()
        opt.step()

        pred = (out>0.5).int()
        correct += (pred==yb.int()).sum().item()
        total += yb.size(0)

    print("Epoch",epoch+1,"Accuracy:",correct/total)

Epoch 1 Accuracy: 0.7755344418052257
Epoch 2 Accuracy: 0.9477434679334917
Epoch 3 Accuracy: 0.9501187648456056
Epoch 4 Accuracy: 0.9501187648456056
Epoch 5 Accuracy: 0.9501187648456056


In [24]:
with torch.no_grad():
    out = model(X_test.to(device))
    pred = (out>0.5).int().squeeze()

    acc = (pred.cpu()==y_test).sum()/len(y_test)

    print("Test Accuracy:",acc.item())

Test Accuracy: 0.8388625383377075


In [25]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict(text):
    model.eval()

    # Step 1: convert text → sequence
    seq = tokenizer.texts_to_sequences([text])

    # Step 2: pad
    seq = pad_sequences(seq, maxlen=MAX_LEN)

    # Step 3: tensor
    t = torch.tensor(seq).to(device)

    # Step 4: prediction
    with torch.no_grad():
        out = model(t).item()

    # Step 5: output
    print("Review:", text)
    print("Score:", round(out,4))

    if out > 0.5:
        print("Sentiment: Positive ✅")
    else:
        print("Sentiment: Negative ❌")

In [27]:
predict("this product is amazing and worth buying")
predict("average product")

Review: this product is amazing and worth buying
Score: 0.948
Sentiment: Positive ✅
Review: average product
Score: 0.9252
Sentiment: Positive ✅
